In [ ]:
# 使用内存存储

In [2]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.chat_models import init_chat_model
import os, dotenv

dotenv.load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

# temperature 从低到高 越大越随机
dashscope_chat = init_chat_model(
    model='openai:qwen3.7-max',
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    temperature=0,
    max_tokens=2000,
    max_retries=10
)
# 存储每个历史对话
store = {}


def get_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 包装大模型，自动注入历史
chain = RunnableWithMessageHistory(
    dashscope_chat,
    get_history,
)

# 使用
from langchain_core.messages import SystemMessage, HumanMessage

config = {"configurable": {"session_id": "user_001"}}
# 第一轮对话
resp1 = chain.invoke([
    SystemMessage(content="你是善解人意的助手"),
    HumanMessage(content="你好"),
], config=config)
print(resp1.content)


/Users/zhouhaijun/anaconda3/envs/LangChain_Test/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


你好呀！很高兴能和你交流。今天过得还好吗？

有什么我可以帮你的，或者只是想随便聊聊、分享一下今天的心情，都可以告诉我哦。我随时都在这里听你说。


In [3]:
# 第二轮对话
resp2 = chain.invoke([
    HumanMessage(content="我刚才说了什么")],
    config=config)
print(resp2.content)

你刚才说的是“你好”呀。😊 

怎么啦，是突然走神忘了，还是在考考我的记忆力呢？不管你想聊什么，或者有什么心事，我都在这里陪着你哦。


In [4]:
# 添加提示词模板
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 添加提示词
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}，请用{style}的风格回答"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 组合chain
chain1 = prompt | dashscope_chat
chain_with_memory = RunnableWithMessageHistory(
    chain1,
    lambda sid: store.setdefault(sid, ChatMessageHistory()),
    input_messages_key="question",
    history_messages_key="history",
)
config = {"configurable": {"session_id": "user_001"}}
resp = chain_with_memory.invoke(
    {"role": "法律顾问", "style": "专业", "question": "租房要注意什么？"},
    config=config
)
print(resp.content)

/Users/zhouhaijun/anaconda3/envs/LangChain_Test/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


您好。作为您的法律顾问，我非常理解租房过程中的繁琐，以及您对于保障自身权益、避免“踩坑”的关切。

为了最大程度保障您的合法权益，防范潜在的法律纠纷，我建议您从以下四个核心法律维度进行审查和把控：

### 一、 缔约前的审查：确认“人”与“房”的合法性

1. **核实出租人主体资格（防范无权处分）**
   * **房东直租**：务必查验其身份证原件及《不动产权证书》（或房产证）原件，确保“人证合一”。
   * **“二房东”转租**：必须要求其提供原房东**同意转租的书面证明**以及原租赁合同。根据《民法典》第七百一十六条，未经出租人同意的转租，原房东有权解除合同，这将导致您面临被随时清退的风险。
2. **审查房屋权利状态与性质**
   * 确认房屋是否存在抵押、查封情况。若房屋在出租前已设立抵押权，未来抵押权实现（如房屋被拍卖）时，可能无法适用“买卖不破租赁”原则。
   * 确认房屋性质，避免租赁违章建筑、违规隔断房或擅自改变房屋用途（如商改住、工业改住），此类租赁合同在法律上可能被认定为无效。

### 二、 缔约时的把控：完善租赁合同核心条款

1. **明确费用与押金退还条件**
   * 详细列明租金、押金数额及支付周期。
   * **重点防范**：明确押金退还的具体条件和时间。避免使用“房屋无损即退”等模糊表述，建议在合同中明确约定 **“房屋及设施的自然损耗不扣除押金”**。
2. **界定维修与养护责任**
   * 根据《民法典》第七百一十二条，出租人应当履行租赁物的维修义务，但当事人另有约定的除外。
   * 建议在合同中明确：房屋主体结构及原有家电家具的自然老化、损坏由房东负责维修；若房东怠于维修，租客可自行维修，**费用从租金中扣除或由房东承担**。
3. **违约责任与解除权**
   * 明确双方单方解除合同的条件及违约金计算标准。警惕“霸王条款”，例如“房东可随时无条件收回房屋”或“租客逾期交租一日即没收全额押金”等显失公平的条款。
4. **转租与续租条款**
   * 明确您在租期内是否拥有转租权。
   * 根据《民法典》第七百三十四条，租赁期限届满，房屋承租人享有
